In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import (
    make_scorer,
    matthews_corrcoef,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)


import sys
sys.path.append("../../utils/")

from utils import *

/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ===== RUTAS =====
PROJECT_ROOT = Path.cwd().resolve().parents[2]

NOMBRE_EXPERIMENTO = "CIC17__NearMiss_SMOTE__v1__pca4_mlp__v1"
CARPETA_DATASET = "CIC17__NearMiss_SMOTE__v1"

NOMBRE_DATASET_TRAIN = f"{CARPETA_DATASET}__train.csv"
NOMBRE_DATASET_TEST = f"{CARPETA_DATASET}__test.csv"

RUTA_DATASET = PROJECT_ROOT / "02_datasets" / "processed" / CARPETA_DATASET
RUTA_RESULTADOS = PROJECT_ROOT / "04_experimentos" / "logs" / "resultados" / NOMBRE_EXPERIMENTO

NOMBRE_RESULTADOS_CV_CSV = f"{NOMBRE_EXPERIMENTO}__folds.csv"
NOMBRE_RESULTADOS_CV_JSON = f"{NOMBRE_EXPERIMENTO}__summary_cv.json"
NOMBRE_RESULTADOS_TEST_JSON = f"{NOMBRE_EXPERIMENTO}__summary_test.json"
NOMBRE_RESULTADOS_TEST_CSV = f"{NOMBRE_EXPERIMENTO}__metricas_test.csv"
NOMBRE_RESULTADOS_TEST_CM_CSV = f"{NOMBRE_EXPERIMENTO}__confusion_matrix_test.csv"

# ===== PARÁMETROS =====
LABEL_COL = "LABEL"

N_SPLITS = 5
SHUFFLE = True
RANDOM_STATE = 42

# ===== CONFIG MLP =====
HIDDEN_LAYER_SIZES = (64, 32)
ACTIVATION = "relu"
SOLVER = "adam"
MAX_ITER = 200

# ===== CONFIG PCA =====
N_COMPONENTS_PCA = 4

In [3]:
RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)

print("Ruta dataset train:")
print((RUTA_DATASET / NOMBRE_DATASET_TRAIN).resolve())
print()

print("Ruta dataset test:")
print((RUTA_DATASET / NOMBRE_DATASET_TEST).resolve())
print()

print("Ruta resultados:")
print(RUTA_RESULTADOS.resolve())

Ruta dataset train:
/home/javier/TFG_MODELOS_SIMPLES/02_datasets/processed/CIC17__NearMiss_SMOTE__v1/CIC17__NearMiss_SMOTE__v1__train.csv

Ruta dataset test:
/home/javier/TFG_MODELOS_SIMPLES/02_datasets/processed/CIC17__NearMiss_SMOTE__v1/CIC17__NearMiss_SMOTE__v1__test.csv

Ruta resultados:
/home/javier/TFG_MODELOS_SIMPLES/04_experimentos/logs/resultados/CIC17__NearMiss_SMOTE__v1__pca4_mlp__v1


In [4]:
df_train = cargar_dataset(
    nombre_dataset=NOMBRE_DATASET_TRAIN,
    ruta_base=RUTA_DATASET
)

print("Forma del dataset train:")
print(df_train.shape)

df_train.head()

Forma del dataset train:
(150000, 48)


,DESTINATION_PORT,FLOW_DURATION,TOTAL_FWD_PACKETS,TOTAL_LENGTH_OF_FWD_PACKETS,FWD_PACKET_LENGTH_MAX,FWD_PACKET_LENGTH_MIN,FWD_PACKET_LENGTH_MEAN,BWD_PACKET_LENGTH_MAX,BWD_PACKET_LENGTH_MIN,FLOW_BYTES_S,...,INIT_WIN_BYTES_FORWARD,INIT_WIN_BYTES_BACKWARD,ACT_DATA_PKT_FWD,MIN_SEG_SIZE_FORWARD,ACTIVE_MEAN,ACTIVE_STD,ACTIVE_MAX,ACTIVE_MIN,IDLE_STD,LABEL
0,443,119985408,257,224020,14628,0,871.673152,1075,0,2373.255255,...,258,2043,124,20,0.0,0.0,0,0,0.0,0
1,443,118709021,3141,111133,944,0,35.381407,4549,2,71774.899060,...,3165,348,118,20,0.0,0.0,0,0,0.0,0
2,443,118451558,18605,156557,486,0,8.414781,4380,6,471911.437400,...,7294,437,18591,20,0.0,0.0,0,0,0.0,0
3,443,118190125,2743,113004,944,0,41.197229,4484,0,62152.561390,...,3758,348,120,20,0.0,0.0,0,0,0.0,0
4,443,118034742,18494,155286,486,0,8.396561,8760,6,470791.430200,...,3798,437,18302,20,0.0,0.0,0,0,0.0,0


In [5]:
if LABEL_COL not in df_train.columns:
    raise ValueError(f"No se encontró la columna {LABEL_COL} en train")

print("Última columna train:", df_train.columns[-1])
print("Tipo de LABEL train:", df_train[LABEL_COL].dtype)
print()

print("Distribución de clases en train:")
display(df_train[LABEL_COL].value_counts(dropna=False).to_frame("count"))

Última columna train: LABEL
Tipo de LABEL train: int64

Distribución de clases en train:


,count
LABEL,
0,10000
1,10000
2,10000
3,10000
4,10000
5,10000
6,10000
7,10000
8,10000


In [6]:
X_train = df_train.drop(columns=[LABEL_COL]).copy()
y_train = df_train[LABEL_COL].copy()

print("Shape X_train:", X_train.shape)
print("Shape y_train:", y_train.shape)

Shape X_train: (150000, 47)
Shape y_train: (150000,)


In [7]:
columnas_no_numericas_train = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

print("Columnas no numéricas en X_train:")
print(columnas_no_numericas_train)

if len(columnas_no_numericas_train) > 0:
    raise ValueError("Hay columnas no numéricas en X_train. Revísalas antes de seguir.")

Columnas no numéricas en X_train:
[]


In [8]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=N_COMPONENTS_PCA)),
    ("model", MLPClassifier(
        hidden_layer_sizes=HIDDEN_LAYER_SIZES,
        activation=ACTIVATION,
        solver=SOLVER,
        max_iter=MAX_ITER,
        early_stopping=True,
        random_state=RANDOM_STATE
    ))
])

pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('pca', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"n_components n_components: int, float or 'mle', default=NoneNumber of components to keep.if n_components is not set all components are kept:: n_components == min(n_samples, n_features)If ``n_components == 'mle'`` and ``svd_solver == 'full'``, Minka'sMLE is used to guess the dimension. Use of ``n_components == 'mle'``will interpret ``svd_solver == 'auto'`` as ``svd_solver == 'full'``.If ``0 < n_components < 1`` and ``svd_solver == 'full'``, select thenumber of components such that the amount of variance that needs to beexplained is greater than the percentage specified by n_components.If ``svd_solver == 'arpack'``, the number of components must bestrictly less than the minimum of n_features and n_samples.Hence, the None case results in:: n_components == min(n_samples, n_features) - 1",4
,"copy copy: bool, default=TrueIf False, data passed to fit are overwritten and runningfit(X).transform(X) will not yield the expected results,use fit_transform(X) instead.",True
,"whiten whiten: bool, default=FalseWhen True (False by default) the `components_` vectors are multipliedby the square root of n_samples and then divided by the singular valuesto ensure uncorrelated outputs with unit component-wise variances.Whitening will remove some information from the transformed signal(the relative variance scales of the components) but can sometimeimprove the predictive accuracy of the downstream estimators bymaking their data respect some hard-wired assumptions.",False
,"svd_solver svd_solver: {'auto', 'fu

In [9]:
cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=SHUFFLE,
    random_state=RANDOM_STATE
)

cv

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

In [10]:
scoring = {
    "accuracy": "accuracy",

    "precision_weighted": "precision_weighted",
    "recall_weighted": "recall_weighted",
    "f1_weighted": "f1_weighted",

    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro",

    "mcc": make_scorer(matthews_corrcoef),

    "roc_auc": "roc_auc_ovr_weighted"

}

In [11]:
cv_results = cross_validate(
    estimator=pipeline,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=-1
)

cv_results.keys()

dict_keys(['fit_time', 'score_time', 'test_accuracy', 'test_precision_weighted', 'test_recall_weighted', 'test_f1_weighted', 'test_precision_macro', 'test_recall_macro', 'test_f1_macro', 'test_mcc', 'test_roc_auc'])

In [12]:
df_folds = pd.DataFrame({
    "fold": np.arange(1, N_SPLITS + 1),

    "accuracy": cv_results["test_accuracy"],

    "precision_weighted": cv_results["test_precision_weighted"],
    "recall_weighted": cv_results["test_recall_weighted"],
    "f1_weighted": cv_results["test_f1_weighted"],

    "precision_macro": cv_results["test_precision_macro"],
    "recall_macro": cv_results["test_recall_macro"],
    "f1_macro": cv_results["test_f1_macro"],

    "mcc": cv_results["test_mcc"],
    "roc_auc": cv_results["test_roc_auc"],

    "fit_time": cv_results["fit_time"],
    "score_time": cv_results["score_time"]
})

df_folds

,fold,accuracy,precision_weighted,recall_weighted,f1_weighted,precision_macro,recall_macro,f1_macro,mcc,roc_auc,fit_time,score_time
0,1,0.934900,0.935861,0.934900,0.933554,0.935861,0.934900,0.933554,0.930585,0.995760,11.469852,0.312778
1,2,0.936600,0.938800,0.936600,0.934446,0.938800,0.936600,0.934446,0.932657,0.995900,14.234837,0.385270
2,3,0.939867,0.941953,0.939867,0.937312,0.941953,0.939867,0.937312,0.936208,0.996161,21.551343,0.294832
3,4,0.934567,0.937008,0.934567,0.931870,0.937008,0.934567,0.931870,0.930589,0.995905,14.141282,0.224447
4,5,0.933933,0.938727,0.933933,0.929677,0.938727,0.933933,0.929677,0.930352,0.995719,16.439990,0.235790


In [13]:
summary_cv = {
    "experimento": NOMBRE_EXPERIMENTO,
    "dataset": str(RUTA_DATASET / NOMBRE_DATASET_TRAIN),
    "shape_dataset": {
        "rows": int(df_train.shape[0]),
        "cols": int(df_train.shape[1])
    },
    "parametros": {
        "label_col": LABEL_COL,
        "n_splits": N_SPLITS,
        "shuffle": SHUFFLE,
        "random_state": RANDOM_STATE,
        "n_components_pca": N_COMPONENTS_PCA,
        "modelo": "MLPClassifier",
        "hidden_layer_sizes": HIDDEN_LAYER_SIZES,
        "activation": ACTIVATION,
        "solver": SOLVER,
        "max_iter": MAX_ITER,
        "dataset_balanceado": CARPETA_DATASET
    },
    "metricas_media": {
        "accuracy": float(df_folds["accuracy"].mean()),

        "precision_weighted": float(df_folds["precision_weighted"].mean()),
        "recall_weighted": float(df_folds["recall_weighted"].mean()),
        "f1_weighted": float(df_folds["f1_weighted"].mean()),

        "precision_macro": float(df_folds["precision_macro"].mean()),
        "recall_macro": float(df_folds["recall_macro"].mean()),
        "f1_macro": float(df_folds["f1_macro"].mean()),

        "mcc": float(df_folds["mcc"].mean()),
        "roc_auc": float(df_folds["roc_auc"].mean()),
        "fit_time": float(df_folds["fit_time"].mean()),
        "score_time": float(df_folds["score_time"].mean())
    },
    "metricas_std": {
        "accuracy": float(df_folds["accuracy"].std(ddof=1)),

        "precision_weighted": float(df_folds["precision_weighted"].std(ddof=1)),
        "recall_weighted": float(df_folds["recall_weighted"].std(ddof=1)),
        "f1_weighted": float(df_folds["f1_weighted"].std(ddof=1)),

        "precision_macro": float(df_folds["precision_macro"].std(ddof=1)),
        "recall_macro": float(df_folds["recall_macro"].std(ddof=1)),
        "f1_macro": float(df_folds["f1_macro"].std(ddof=1)),

        "mcc": float(df_folds["mcc"].std(ddof=1)),
        "roc_auc": float(df_folds["roc_auc"].std(ddof=1)),
        "fit_time": float(df_folds["fit_time"].std(ddof=1)),
        "score_time": float(df_folds["score_time"].std(ddof=1))
    }
}

summary_cv

{'experimento': 'CIC17__NearMiss_SMOTE__v1__pca4_mlp__v1',
 'dataset': '/home/javier/TFG_MODELOS_SIMPLES/02_datasets/processed/CIC17__NearMiss_SMOTE__v1/CIC17__NearMiss_SMOTE__v1__train.csv',
 'shape_dataset': {'rows': 150000, 'cols': 48},
 'parametros': {'label_col': 'LABEL',
  'n_splits': 5,
  'shuffle': True,
  'random_state': 42,
  'n_components_pca': 4,
  'modelo': 'MLPClassifier',
  'hidden_layer_sizes': (64, 32),
  'activation': 'relu',
  'solver': 'adam',
  'max_iter': 200,
  'dataset_balanceado': 'CIC17__NearMiss_SMOTE__v1'},
 'metricas_media': {'accuracy': 0.9359733333333333,
  'precision_weighted': 0.9384697103399781,
  'recall_weighted': 0.9359733333333333,
  'f1_weighted': 0.9333717838318958,
  'precision_macro': 0.9384697103399782,
  'recall_macro': 0.9359733333333334,
  'f1_macro': 0.9333717838318958,
  'mcc': 0.9320780816586124,
  'roc_auc': 0.9958891783333333,
  'fit_time': 15.567460870742797,
  'score_time': 0.2906233787536621},
 'metricas_std': {'accuracy': 0.0023897

In [14]:
print("========== RESULTADOS CV ==========")
print(f"Accuracy            : {summary_cv['metricas_media']['accuracy']:.6f} ± {summary_cv['metricas_std']['accuracy']:.6f}")
print()

print(f"Precision weighted  : {summary_cv['metricas_media']['precision_weighted']:.6f} ± {summary_cv['metricas_std']['precision_weighted']:.6f}")
print(f"Recall weighted     : {summary_cv['metricas_media']['recall_weighted']:.6f} ± {summary_cv['metricas_std']['recall_weighted']:.6f}")
print(f"F1 weighted         : {summary_cv['metricas_media']['f1_weighted']:.6f} ± {summary_cv['metricas_std']['f1_weighted']:.6f}")
print()

print(f"Precision macro     : {summary_cv['metricas_media']['precision_macro']:.6f} ± {summary_cv['metricas_std']['precision_macro']:.6f}")
print(f"Recall macro        : {summary_cv['metricas_media']['recall_macro']:.6f} ± {summary_cv['metricas_std']['recall_macro']:.6f}")
print(f"F1 macro            : {summary_cv['metricas_media']['f1_macro']:.6f} ± {summary_cv['metricas_std']['f1_macro']:.6f}")
print()

print(f"MCC                 : {summary_cv['metricas_media']['mcc']:.6f} ± {summary_cv['metricas_std']['mcc']:.6f}")
print(f"ROC AUC              : {summary_cv['metricas_media']['roc_auc']:.6f} ± {summary_cv['metricas_std']['roc_auc']:.6f}")
print()
print(f"Fit time medio      : {summary_cv['metricas_media']['fit_time']:.6f}")
print(f"Score time medio    : {summary_cv['metricas_media']['score_time']:.6f}")

========== RESULTADOS CV ==========
Accuracy            : 0.935973 ± 0.002390

Precision weighted  : 0.938470 ± 0.002305
Recall weighted     : 0.935973 ± 0.002390
F1 weighted         : 0.933372 ± 0.002855

Precision macro     : 0.938470 ± 0.002305
Recall macro        : 0.935973 ± 0.002390
F1 macro            : 0.933372 ± 0.002855

MCC                 : 0.932078 ± 0.002491
ROC AUC              : 0.995889 ± 0.000173

Fit time medio      : 15.567461
Score time medio    : 0.290623


In [15]:
ruta_cv_csv = RUTA_RESULTADOS / NOMBRE_RESULTADOS_CV_CSV
df_folds.to_csv(ruta_cv_csv, index=False)

print("Resultados por fold guardados en:")
print(ruta_cv_csv.resolve())

Resultados por fold guardados en:
/home/javier/TFG_MODELOS_SIMPLES/04_experimentos/logs/resultados/CIC17__NearMiss_SMOTE__v1__pca4_mlp__v1/CIC17__NearMiss_SMOTE__v1__pca4_mlp__v1__folds.csv


In [16]:
ruta_cv_json = RUTA_RESULTADOS / NOMBRE_RESULTADOS_CV_JSON

with open(ruta_cv_json, "w", encoding="utf-8") as f:
    json.dump(summary_cv, f, indent=4, ensure_ascii=False)

print("Resumen CV guardado en:")
print(ruta_cv_json.resolve())

Resumen CV guardado en:
/home/javier/TFG_MODELOS_SIMPLES/04_experimentos/logs/resultados/CIC17__NearMiss_SMOTE__v1__pca4_mlp__v1/CIC17__NearMiss_SMOTE__v1__pca4_mlp__v1__summary_cv.json


In [17]:
df_folds

,fold,accuracy,precision_weighted,recall_weighted,f1_weighted,precision_macro,recall_macro,f1_macro,mcc,roc_auc,fit_time,score_time
0,1,0.934900,0.935861,0.934900,0.933554,0.935861,0.934900,0.933554,0.930585,0.995760,11.469852,0.312778
1,2,0.936600,0.938800,0.936600,0.934446,0.938800,0.936600,0.934446,0.932657,0.995900,14.234837,0.385270
2,3,0.939867,0.941953,0.939867,0.937312,0.941953,0.939867,0.937312,0.936208,0.996161,21.551343,0.294832
3,4,0.934567,0.937008,0.934567,0.931870,0.937008,0.934567,0.931870,0.930589,0.995905,14.141282,0.224447
4,5,0.933933,0.938727,0.933933,0.929677,0.938727,0.933933,0.929677,0.930352,0.995719,16.439990,0.235790


In [18]:
df_test = cargar_dataset(
    nombre_dataset=NOMBRE_DATASET_TEST,
    ruta_base=RUTA_DATASET
)

print("Forma del dataset test:")
print(df_test.shape)

df_test.head()

Forma del dataset test:
(504160, 48)


,DESTINATION_PORT,FLOW_DURATION,TOTAL_FWD_PACKETS,TOTAL_LENGTH_OF_FWD_PACKETS,FWD_PACKET_LENGTH_MAX,FWD_PACKET_LENGTH_MIN,FWD_PACKET_LENGTH_MEAN,BWD_PACKET_LENGTH_MAX,BWD_PACKET_LENGTH_MIN,FLOW_BYTES_S,...,INIT_WIN_BYTES_FORWARD,INIT_WIN_BYTES_BACKWARD,ACT_DATA_PKT_FWD,MIN_SEG_SIZE_FORWARD,ACTIVE_MEAN,ACTIVE_STD,ACTIVE_MAX,ACTIVE_MIN,IDLE_STD,LABEL
0,443,93606787,18,3128,410,6,173.777778,1618,38,186.845426,...,256,8192,17,20,32068.8000,1749.725464,34107,30651,6.931172e+06,0
1,80,98331732,6,354,336,0,59.000000,4344,0,121.517233,...,0,235,3,20,21035.0000,0.000000,21035,21035,0.000000e+00,1
2,80,117175775,231,1427,401,0,6.177489,4584,0,5590.916723,...,29200,1039,4,32,206922.5455,636207.069100,2125159,14988,8.932177e+04,0
3,80,1285910,3,566,560,0,188.666667,1894,2,1923.929357,...,29200,15680,2,20,0.0000,0.000000,0,0,0.000000e+00,0
4,53,47941,1,44,44,44,44.000000,224,224,5590.204627,...,-1,-1,0,40,0.0000,0.000000,0,0,0.000000e+00,0


In [19]:
if LABEL_COL not in df_test.columns:
    raise ValueError(f"No se encontró la columna {LABEL_COL} en test")

print("Última columna test:", df_test.columns[-1])
print("Tipo de LABEL test:", df_test[LABEL_COL].dtype)
print()

print("Distribución de clases en test:")
display(df_test[LABEL_COL].value_counts(dropna=False).to_frame("count"))

Última columna test: LABEL
Tipo de LABEL test: int64

Distribución de clases en test:


,count
LABEL,
0,419012
1,34569
2,25603
3,18139
4,2057
5,1186
6,1077
7,1046
8,644


In [20]:
X_test = df_test.drop(columns=[LABEL_COL]).copy()
y_test = df_test[LABEL_COL].copy()

print("Shape X_test:", X_test.shape)
print("Shape y_test:", y_test.shape)

Shape X_test: (504160, 47)
Shape y_test: (504160,)


In [21]:
columnas_no_numericas_test = X_test.select_dtypes(exclude=[np.number]).columns.tolist()

print("Columnas no numéricas en X_test:")
print(columnas_no_numericas_test)

if len(columnas_no_numericas_test) > 0:
    raise ValueError("Hay columnas no numéricas en X_test. Revísalas antes de seguir.")

Columnas no numéricas en X_test:
[]


In [22]:
pipeline.fit(X_train, y_train)

print("Modelo final entrenado con todo el dataset train.")

Modelo final entrenado con todo el dataset train.


In [23]:
y_pred_test = pipeline.predict(X_test)

print("Predicciones en test generadas.")
print("Número de predicciones:", len(y_pred_test))

y_proba_test = pipeline.predict_proba(X_test)

roc_auc_test = roc_auc_score(
    y_test,
    y_proba_test,
    multi_class="ovr",
    average="weighted"
)

Predicciones en test generadas.
Número de predicciones: 504160


In [24]:
metricas_test = {
    "accuracy": accuracy_score(y_test, y_pred_test),

    "precision_weighted": precision_score(y_test, y_pred_test, average="weighted", zero_division=0),
    "recall_weighted": recall_score(y_test, y_pred_test, average="weighted", zero_division=0),
    "f1_weighted": f1_score(y_test, y_pred_test, average="weighted", zero_division=0),

    "precision_macro": precision_score(y_test, y_pred_test, average="macro", zero_division=0),
    "recall_macro": recall_score(y_test, y_pred_test, average="macro", zero_division=0),
    "f1_macro": f1_score(y_test, y_pred_test, average="macro", zero_division=0),

    "mcc": matthews_corrcoef(y_test, y_pred_test),
    "roc_auc": roc_auc_test,
}

metricas_test

{'accuracy': 0.16648087908600445,
 'precision_weighted': 0.8100354404197078,
 'recall_weighted': 0.16648087908600445,
 'f1_weighted': 0.1662965671681024,
 'precision_macro': 0.17201737930801625,
 'recall_macro': 0.7348978029718103,
 'f1_macro': 0.1527442005177321,
 'mcc': 0.19866651124256438,
 'roc_auc': 0.6030533508740408}

In [25]:
print("========== RESULTADOS TEST ==========")
print(f"Accuracy            : {metricas_test['accuracy']:.6f}")
print()

print(f"Precision weighted  : {metricas_test['precision_weighted']:.6f}")
print(f"Recall weighted     : {metricas_test['recall_weighted']:.6f}")
print(f"F1 weighted         : {metricas_test['f1_weighted']:.6f}")
print()

print(f"Precision macro     : {metricas_test['precision_macro']:.6f}")
print(f"Recall macro        : {metricas_test['recall_macro']:.6f}")
print(f"F1 macro            : {metricas_test['f1_macro']:.6f}")
print()

print(f"MCC                 : {metricas_test['mcc']:.6f}")
print(f"ROC AUC              : {metricas_test['roc_auc']:.6f}")


========== RESULTADOS TEST ==========
Accuracy            : 0.166481

Precision weighted  : 0.810035
Recall weighted     : 0.166481
F1 weighted         : 0.166297

Precision macro     : 0.172017
Recall macro        : 0.734898
F1 macro            : 0.152744

MCC                 : 0.198667
ROC AUC              : 0.603053


In [26]:
labels_ordenadas = sorted(pd.unique(pd.concat([y_test, pd.Series(y_pred_test)])))

cm = confusion_matrix(y_test, y_pred_test, labels=labels_ordenadas)
df_cm = pd.DataFrame(cm, index=labels_ordenadas, columns=labels_ordenadas)

print("Matriz de confusión en test:")
display(df_cm)

Matriz de confusión en test:


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
0,26440,55295,5713,28381,15039,12002,5350,14207,16401,41129,22870,12933,144647,18546,59
1,70,26241,0,1,4942,0,20,264,34,64,698,1429,798,7,1
2,2862,10961,6648,4,0,82,0,6,0,12,0,0,3742,0,1286
3,0,9,0,18079,0,0,4,0,0,0,0,47,0,0,0
4,0,1,0,0,2025,0,0,4,18,0,1,0,0,8,0
5,0,0,0,0,0,1171,10,0,0,0,1,2,0,2,0
6,0,0,0,1,0,0,1061,8,3,0,0,1,1,2,0
7,0,0,0,0,0,0,5,1040,0,0,0,0,0,1,0
8,0,0,0,5,0,4,0,1,604,0,0,1,0,29,0
9,0,0,0,0,0,0,0,0,0,390,0,0,0,0,0


In [27]:
print("========== CLASSIFICATION REPORT TEST ==========")
print(classification_report(y_test, y_pred_test, zero_division=0))

========== CLASSIFICATION REPORT TEST ==========


              precision    recall  f1-score   support

           0       0.90      0.06      0.12    419012
           1       0.28      0.76      0.41     34569
           2       0.54      0.26      0.35     25603
           3       0.39      1.00      0.56     18139
           4       0.09      0.98      0.17      2057
           5       0.09      0.99      0.16      1186
           6       0.16      0.99      0.28      1077
           7       0.07      0.99      0.13      1046
           8       0.04      0.94      0.07       644
           9       0.01      1.00      0.02       390
          10       0.01      0.42      0.01       294
          11       0.01      0.78      0.01       130
          12       0.00      0.86      0.00         7
          13       0.00      0.50      0.00         4
          14       0.00      0.50      0.00         2

    accuracy                           0.17    504160
   macro avg       0.17      0.73      0.15    504160
weighted avg       0.81   

In [28]:
summary_test = {
    "experimento": NOMBRE_EXPERIMENTO,
    "dataset_test": str(RUTA_DATASET / NOMBRE_DATASET_TEST),
    "shape_test": {
        "rows": int(df_test.shape[0]),
        "cols": int(df_test.shape[1])
    },
    "parametros": {
        "label_col": LABEL_COL,
        "n_splits": N_SPLITS,
        "shuffle": SHUFFLE,
        "random_state": RANDOM_STATE,
        "n_components_pca": N_COMPONENTS_PCA,
        "modelo": "MLPClassifier",
        "hidden_layer_sizes": HIDDEN_LAYER_SIZES,
        "activation": ACTIVATION,
        "solver": SOLVER,
        "max_iter": MAX_ITER,
        "dataset_balanceado": CARPETA_DATASET
    },
    "metricas_test": {
        "accuracy": float(metricas_test["accuracy"]),

        "precision_weighted": float(metricas_test["precision_weighted"]),
        "recall_weighted": float(metricas_test["recall_weighted"]),
        "f1_weighted": float(metricas_test["f1_weighted"]),

        "precision_macro": float(metricas_test["precision_macro"]),
        "recall_macro": float(metricas_test["recall_macro"]),
        "f1_macro": float(metricas_test["f1_macro"]),

        "mcc": float(metricas_test["mcc"]),
        "roc_auc": float(metricas_test["roc_auc"]),
    }
}

summary_test

{'experimento': 'CIC17__NearMiss_SMOTE__v1__pca4_mlp__v1',
 'dataset_test': '/home/javier/TFG_MODELOS_SIMPLES/02_datasets/processed/CIC17__NearMiss_SMOTE__v1/CIC17__NearMiss_SMOTE__v1__test.csv',
 'shape_test': {'rows': 504160, 'cols': 48},
 'parametros': {'label_col': 'LABEL',
  'n_splits': 5,
  'shuffle': True,
  'random_state': 42,
  'n_components_pca': 4,
  'modelo': 'MLPClassifier',
  'hidden_layer_sizes': (64, 32),
  'activation': 'relu',
  'solver': 'adam',
  'max_iter': 200,
  'dataset_balanceado': 'CIC17__NearMiss_SMOTE__v1'},
 'metricas_test': {'accuracy': 0.16648087908600445,
  'precision_weighted': 0.8100354404197078,
  'recall_weighted': 0.16648087908600445,
  'f1_weighted': 0.1662965671681024,
  'precision_macro': 0.17201737930801625,
  'recall_macro': 0.7348978029718103,
  'f1_macro': 0.1527442005177321,
  'mcc': 0.19866651124256438,
  'roc_auc': 0.6030533508740408}}

In [29]:
df_metricas_test = pd.DataFrame([metricas_test])

ruta_test_csv = RUTA_RESULTADOS / NOMBRE_RESULTADOS_TEST_CSV
df_metricas_test.to_csv(ruta_test_csv, index=False)

print("Métricas test guardadas en:")
print(ruta_test_csv.resolve())

Métricas test guardadas en:
/home/javier/TFG_MODELOS_SIMPLES/04_experimentos/logs/resultados/CIC17__NearMiss_SMOTE__v1__pca4_mlp__v1/CIC17__NearMiss_SMOTE__v1__pca4_mlp__v1__metricas_test.csv


In [30]:
ruta_cm_csv = RUTA_RESULTADOS / NOMBRE_RESULTADOS_TEST_CM_CSV
df_cm.to_csv(ruta_cm_csv, index=True)

print("Matriz de confusión test guardada en:")
print(ruta_cm_csv.resolve())

Matriz de confusión test guardada en:
/home/javier/TFG_MODELOS_SIMPLES/04_experimentos/logs/resultados/CIC17__NearMiss_SMOTE__v1__pca4_mlp__v1/CIC17__NearMiss_SMOTE__v1__pca4_mlp__v1__confusion_matrix_test.csv


In [31]:
ruta_test_json = RUTA_RESULTADOS / NOMBRE_RESULTADOS_TEST_JSON

with open(ruta_test_json, "w", encoding="utf-8") as f:
    json.dump(summary_test, f, indent=4, ensure_ascii=False)

print("Resumen test guardado en:")
print(ruta_test_json.resolve())

Resumen test guardado en:
/home/javier/TFG_MODELOS_SIMPLES/04_experimentos/logs/resultados/CIC17__NearMiss_SMOTE__v1__pca4_mlp__v1/CIC17__NearMiss_SMOTE__v1__pca4_mlp__v1__summary_test.json


In [32]:
print("========== RESUMEN FINAL ==========")
print("CV:")
print(summary_cv["metricas_media"])
print()
print("TEST:")
print(summary_test["metricas_test"])

========== RESUMEN FINAL ==========
CV:
{'accuracy': 0.9359733333333333, 'precision_weighted': 0.9384697103399781, 'recall_weighted': 0.9359733333333333, 'f1_weighted': 0.9333717838318958, 'precision_macro': 0.9384697103399782, 'recall_macro': 0.9359733333333334, 'f1_macro': 0.9333717838318958, 'mcc': 0.9320780816586124, 'roc_auc': 0.9958891783333333, 'fit_time': 15.567460870742797, 'score_time': 0.2906233787536621}

TEST:
{'accuracy': 0.16648087908600445, 'precision_weighted': 0.8100354404197078, 'recall_weighted': 0.16648087908600445, 'f1_weighted': 0.1662965671681024, 'precision_macro': 0.17201737930801625, 'recall_macro': 0.7348978029718103, 'f1_macro': 0.1527442005177321, 'mcc': 0.19866651124256438, 'roc_auc': 0.6030533508740408}
